# Cleaning and Preprocessing of IMU data

Accelerometer: measures linear acc along the 3 axes.
Gyroscope: measures angular rotation along the 3 axes.

In [11]:
# install requirements
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import scikit as sklearn
from scipy import signal

In [19]:
# function to generate mock IMU data in a CSV (6-axes)
# input: none
# output: csv of data
def generate_mock_imu_csv(
    out_path="mock_imu_6axis.csv",
    duration_s=10,
    sample_rate_hz=100,
    noise_std_acc=0.15,
    noise_std_gyro=0.8,
    seed=42,
):
    """
    Generate synthetic 6-axis IMU data and save to CSV.

    Columns:
      - timestamp_s
      - ax_g, ay_g, az_g
      - gx_dps, gy_dps, gz_dps

    accel units: g
    gyro units: deg/s
    """
    rng = np.random.default_rng(seed)

    n = int(duration_s * sample_rate_hz)
    t = np.arange(n) / sample_rate_hz

    # Base rhythmic motion (roughly punch-like bursts)
    burst = 1.0 + 0.6 * np.sin(2 * np.pi * 1.8 * t)

    # Accelerometer signals (in g)
    ax = 0.25 * np.sin(2 * np.pi * 3.0 * t) * burst + rng.normal(0, noise_std_acc, n)
    ay = 0.18 * np.sin(2 * np.pi * 2.2 * t + 0.8) * burst + rng.normal(0, noise_std_acc, n)
    az = 1.0 + 0.12 * np.sin(2 * np.pi * 1.2 * t + 1.3) + rng.normal(0, noise_std_acc, n)

    # Gyroscope signals (in deg/s)
    gx = 85 * np.sin(2 * np.pi * 2.8 * t) * burst + rng.normal(0, noise_std_gyro, n)
    gy = 70 * np.sin(2 * np.pi * 2.0 * t + 0.4) * burst + rng.normal(0, noise_std_gyro, n)
    gz = 95 * np.sin(2 * np.pi * 3.4 * t + 1.0) * burst + rng.normal(0, noise_std_gyro, n)

    df = pd.DataFrame(
        {
            "timestamp_s": t,
            "ax_g": ax,
            "ay_g": ay,
            "az_g": az,
            "gx_dps": gx,
            "gy_dps": gy,
            "gz_dps": gz,
        }
    )

    df.to_csv(out_path, index=False)
    print(f"Saved {len(df)} rows to {out_path}")
    return df


# # Example usage
# mock_df = generate_mock_imu_csv(out_path="mock_imu_6axis.csv", duration_s=12, sample_rate_hz=100)
# mock_df.head()



## Filtering

- IMUs maybe sensitive to outside stuff called noise. There is some normal band of freqeuncies we care about especially for punches, that frequencies outside of that can be zeroed out to help us focus on the main frequencies that we have.

- what is the normal range of punch frequncies that we care about?
- how is this filtering done?
- at what point is it done?


In [ ]:
# input: 2D matrix, pandas
# output: 2D matrix
# filters out frequncies above 30 Hz

AXES = ["ax", "ay", "az", "gx", "gy", "gz"]
def lowpass_filter(data, cutoff_hz=30.0, fs=100, order=4):
    """
    Low-pass Butterworth filter. Keeps everything below cutoff_hz,
    removes everything above (noise, vibration).

    cutoff_hz=40.0  →  safe for punch data at 100 Hz
    order=4         →  standard, good rolloff without ringing

    Input:  pd.DataFrame or np.ndarray, shape (n_samples, 6)
    Output: np.ndarray, shape (n_samples, 6)  — same shape, filtered values
    """
    if isinstance(data, pd.DataFrame):
        arr = data[AXES].to_numpy(dtype=np.float64)
    elif isinstance(data, np.ndarray):
        arr = data.astype(np.float64)
    else:
        raise TypeError(f"Expected DataFrame or ndarray, got {type(data)}")

    nyq = fs / 2.0
    b, a = signal.butter(order, cutoff_hz / nyq, btype='low')
    return signal.filtfilt(b, a, arr, axis=0)

    

## Magnitude?

- we can also add info of the magnitude of the punch (sqrt(x2 + y2 + z2))
- might be optional dependign on the model we choose

In [20]:
# input: data frame, 2D matrix
# output: 2D matrix
# adds magnitude info

def add_magnitude_info(
    data,
    accel_cols=("ax_g", "ay_g", "az_g"),
    gyro_cols=("gx_dps", "gy_dps", "gz_dps"),
    output="dataframe",
):
    """
    Add accel/gyro magnitude channels to IMU data.

    Args:
        data: pandas DataFrame, CSV path, or NumPy array with 6 columns.
        accel_cols: accel column names (used for DataFrame/CSV input).
        gyro_cols: gyro column names (used for DataFrame/CSV input).
        output: 'dataframe' or 'array'.

    Returns:
        DataFrame or array with 2 extra channels:
        - accel_mag
        - gyro_mag
    """
    if isinstance(data, str):
        df = pd.read_csv(data)
    elif isinstance(data, pd.DataFrame):
        df = data.copy()
    elif isinstance(data, np.ndarray):
        if data.ndim != 2 or data.shape[1] != 6:
            raise ValueError("NumPy input must have shape (n_samples, 6)")
        accel_mag = np.linalg.norm(data[:, 0:3], axis=1)
        gyro_mag = np.linalg.norm(data[:, 3:6], axis=1)
        out = np.column_stack([data, accel_mag, gyro_mag]).astype(np.float32)
        if output == "array":
            return out
        cols = ["ax_g", "ay_g", "az_g", "gx_dps", "gy_dps", "gz_dps", "accel_mag", "gyro_mag"]
        return pd.DataFrame(out, columns=cols)
    else:
        raise TypeError("data must be a DataFrame, CSV path, or NumPy array")

    needed = list(accel_cols) + list(gyro_cols)
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["accel_mag"] = np.linalg.norm(df[list(accel_cols)].to_numpy(dtype=np.float32), axis=1)
    df["gyro_mag"] = np.linalg.norm(df[list(gyro_cols)].to_numpy(dtype=np.float32), axis=1)


    return df.to_numpy(dtype=np.float32)


# # Usage (this cell): CSV -> DataFrame with added magnitude channels
# imu_df_mag = add_magnitude_info("mock_imu_6axis.csv")
# imu_df_mag[["ax_g", "ay_g", "az_g", "accel_mag", "gx_dps", "gy_dps", "gz_dps", "gyro_mag"]].head()


## Windowing
- We cannot predict a punch from a single timestamp.
- We window the data and treat each window as one example to the model.
- Short windows  →  many samples, little context per sample
- Long windows   →  few samples, rich context per sample
- Overlap helps by:
    (1) generating more training samples → better trained model
    (2) reducing boundary misalignment → punch is less likely to be split
    (3) finer prediction frequency → real-time feels more responsive


In [21]:
# windowing logic

# input: data frame, 2d matrix etc
# output: 3D matrix of windowed data
sampling_frequency = 100.0
def segment(
    data,
    sample_rate,
    window_size,
    hop_size,
    feature_cols=None,
    units="samples",
    return_meta=False,
):
    """
    Segment IMU time-series into overlapping windows.

    Args:
        data: pandas DataFrame OR path to CSV file.
        sample_rate: sampling rate in Hz (used when units='seconds').
        window_size: window length (samples or seconds based on units).
        hop_size: hop length (samples or seconds based on units).
        feature_cols: list of columns to window. If None, uses all numeric columns
                      except timestamp/time-like columns.
        units: 'samples' or 'seconds'.
        return_meta: if True, also return window start/end indices.

    Returns:
        windows: np.ndarray with shape (num_windows, window_len, num_features)
        (optional) meta: DataFrame with window start/end sample indices
    """
    # Load CSV path or use provided DataFrame
    if isinstance(data, str):
        df = pd.read_csv(data)
    elif isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        raise TypeError("data must be a pandas DataFrame or CSV path string")

    # Convert window/hop from seconds to samples if requested
    if units == "seconds":
        window_len = int(round(window_size * sample_rate))
        hop_len = int(round(hop_size * sample_rate))
    elif units == "samples":
        window_len = int(window_size)
        hop_len = int(hop_size)
    else:
        raise ValueError("units must be 'samples' or 'seconds'")

    if window_len <= 0 or hop_len <= 0:
        raise ValueError("window_size and hop_size must be > 0")

    # Auto-select feature columns if none provided
    if feature_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        excluded = {"timestamp", "timestamp_s", "time", "t", "label", "class"}
        feature_cols = [c for c in numeric_cols if c.lower() not in excluded]

    if len(feature_cols) == 0:
        raise ValueError("No feature columns found. Pass feature_cols explicitly.")

    X = df[feature_cols].to_numpy(dtype=np.float32)
    n = X.shape[0]

    if n < window_len:
        empty = np.empty((0, window_len, len(feature_cols)), dtype=np.float32)
        if return_meta:
            return empty, pd.DataFrame(columns=["start_idx", "end_idx"])
        return empty

    starts = np.arange(0, n - window_len + 1, hop_len)
    windows = np.stack([X[s : s + window_len] for s in starts], axis=0)

    if return_meta:
        meta = pd.DataFrame(
            {
                "start_idx": starts,
                "end_idx": starts + window_len - 1,
            }
        )
        return windows, meta

    return windows


# ## windowing with magnitude info
# # Example usage with magnitude (8 channels)
# windows_mag, meta_mag = segment(
#     data=imu_df_mag,
#     sample_rate=100,
#     window_size=1.0,
#     hop_size=0.25,
#     units="seconds",
#     feature_cols=["ax_g", "ay_g", "az_g", "gx_dps", "gy_dps", "gz_dps", "accel_mag", "gyro_mag"],
#     return_meta=True,
# )

# print("window tensor with magnitude shape:", windows_mag.shape)  # (num_windows, window_len, 8)
# meta_mag.head()


## Time Synchronization
- we are not doign this since the microcontroller collects all 6 channels at once


## Normalization 
- accelerometer and gryoscope are on different scales. Normalize the data or scale it.
- scaling helps to remove effect of different ranges.

In [ ]:
# input: 2D matrix
# output: scaler object, training_split_index, scaled_data(2D) 

from sklearn import StandardScaler

def fit_scaler(data, train_ratio=0.8):
    """
    Fits a StandardScaler on the training portion of the data,
    transforms the full dataset, and returns both.

    Parameters
    ----------
    data        : np.ndarray, shape (n_samples, 6)
    train_ratio : float, proportion used to fit scaler (default 0.8)

    Returns
    -------
    scaler        : fitted StandardScaler
    split_idx     : int — index where train ends / test begins
    transformed   : np.ndarray, shape (n_samples, 6) — full scaled data
    """
    if not 0.0 < train_ratio < 1.0:
        raise ValueError(f"train_ratio must be between 0 and 1, got {train_ratio}")

    split_idx = int(len(data) * train_ratio)

    scaler = StandardScaler()
    scaler.fit(data[:split_idx])              # fit on training portion only

    transformed = scaler.transform(data)      # apply to full dataset

    return scaler, split_idx, transformed


# scale the data FIRST
# then compute magnitudes and then segment it.